# Granite 3.2 + Self-Consistency Evaluation for CLARITY

This notebook evaluates IBM Granite 3.2-2B-Instruct with self-consistency on the CLARITY dataset.

## Features:
- Uses Granite 3.2 with reasoning capabilities ()
- Self-consistency with 5 samples and majority voting
- Balanced test data loading (equal samples per label)
- Evaluation on QEvasion test set
- CLARITY submission generation

In [1]:
# Install required packages
!pip install -q transformers torch datasets pandas scikit-learn peft

In [2]:
# Import all necessary modules
import pickle
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    

"""
Granite CLARITY Strategy

Uses IBM Granite 3.2-2B-Instruct with reasoning capabilities for CLARITY classification.
Generates JSON-structured output with reasoning and label prediction.
"""

import json
import re
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# Global variables for Granite model (lazy loading)
_granite_model = None
_granite_tokenizer = None
_granite_device = None


def _get_device():
    """Determine the best available device."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


def _load_granite_model():
    """Lazy load Granite model and tokenizer."""
    global _granite_model, _granite_tokenizer, _granite_device
    
    if _granite_model is None:
        model_name = "ibm-granite/granite-3.2-2b-instruct"
        _granite_device = _get_device()
        
        print(f"Loading Granite model: {model_name} on {_granite_device}")
        _granite_tokenizer = AutoTokenizer.from_pretrained(model_name)
        if _granite_tokenizer.pad_token is None:
            _granite_tokenizer.pad_token = _granite_tokenizer.eos_token
        _granite_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        _granite_model.eval()
        print("Granite model loaded successfully")
    
    return _granite_model, _granite_tokenizer, _granite_device


def call_granite_model(messages, temperature=0.7, max_new_tokens=512):
    """
    Call IBM Granite 3.2-2B-Instruct model with reasoning capabilities.
    
    Args:
        messages: List of message dicts with 'role' and 'content' keys
        temperature: Sampling temperature (default 0.7 for self-consistency)
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        Generated text string (includes reasoning if thinking=True)
    """
    model, tokenizer, device = _load_granite_model()
    
    # Apply chat template with thinking=True for reasoning
    try:
        # Try to use thinking=True if supported by the tokenizer
        # Granite 3.2 supports reasoning via thinking parameter
        try:
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                thinking=True  # Enable reasoning mode
            )
        except TypeError:
            # Fallback if thinking parameter not supported
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
        
    except Exception as e:
        # Fallback: manual formatting
        if isinstance(messages, list) and len(messages) > 0:
            # Try to extract user message
            user_msg = None
            for msg in messages:
                if msg.get("role") == "user":
                    user_msg = msg.get("content", "")
                    break
            
            if user_msg:
                formatted = user_msg
            else:
                formatted = str(messages[-1].get("content", ""))
        else:
            formatted = str(messages)
        
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
    
    # Generate with reasoning
    with torch.no_grad():
        # Granite 3.2 supports thinking via special tokens or generation config
        # We'll use the standard generation and let the model use its reasoning
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return generated_text.strip()


def call_granite_model_batch(messages_list, temperature=0.7, max_new_tokens=512):
    """
    Batched version optimized for 15GB T4 GPU - processes multiple samples at once.
    
    Args:
        messages_list: List of message lists (one per sample)
        temperature: Sampling temperature
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        List of generated text strings
    """
    model, tokenizer, device = _load_granite_model()
    
    # Format all prompts
    formatted_texts = []
    for messages in messages_list:
        try:
            try:
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, thinking=True
                )
            except TypeError:
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            formatted_texts.append(formatted)
        except Exception:
            # Fallback: extract user message
            user_msg = next((m.get("content", "") for m in messages if m.get("role") == "user"), "")
            formatted_texts.append(user_msg)
    
    # Tokenize batch with padding
    inputs = tokenizer(formatted_texts, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # Generate batch
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode each response
    results = []
    for i, output in enumerate(outputs):
        input_len = inputs.input_ids[i].shape[0]
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        results.append(generated.strip())
    
    return results


class GraniteClarityStrategy:
    """Strategy for CLARITY classification using Granite 3.2 with reasoning."""

    name = "granite-clarity"

    def build_prompt(self, question, answer):
        """
        Build prompt for CLARITY classification.
        
        Args:
            question: Interview question string
            answer: Interview answer string
        
        Returns:
            Formatted prompt string
        """
        prompt = f"""You are analyzing political interview answers for clarity classification.

Question: {question}
Answer: {answer}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Provide your reasoning and then classify as one of:
- "Direct Reply": Directly answers the question
- "Direct Non-Reply": Explicitly declines or claims inability to answer
- "Indirect": Evasive, indirect, or partially answers

Respond in JSON format:
{{
  "reasoning": "Your step-by-step analysis...",
  "label": "Direct Reply|Direct Non-Reply|Indirect"
}}"""
        return prompt

    def extract_json(self, text: str):
        """Extract JSON from Granite model response."""
        text = text.strip()

        # Remove markdown code blocks if present
        if text.startswith("```json"):
            text = text[7:]
        elif text.startswith("```"):
            text = text[3:]
        if text.endswith("```"):
            text = text[:-3]

        text = text.strip()

        # Try to find JSON object
        try:
            # First, try direct parsing
            return json.loads(text)
        except json.JSONDecodeError:
            # Try to find JSON object in the text
            json_match = re.search(r'\{[^{}]*"reasoning"[^{}]*"label"[^{}]*\}', text, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group())
                except json.JSONDecodeError:
                    pass

            # Try to find any JSON object
            json_match = re.search(r'\{.*\}', text, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group())
                except json.JSONDecodeError:
                    pass

            # Last resort: try to extract label and reasoning separately
            label_match = re.search(r'"label"\s*:\s*"([^"]+)"', text)
            reasoning_match = re.search(r'"reasoning"\s*:\s*"([^"]+)"', text, re.DOTALL)
            
            if label_match:
                result = {"label": label_match.group(1)}
                if reasoning_match:
                    result["reasoning"] = reasoning_match.group(1)
                else:
                    # Try to extract reasoning without quotes (might be multiline)
                    reasoning_match = re.search(r'"reasoning"\s*:\s*([^,}]+)', text, re.DOTALL)
                    if reasoning_match:
                        result["reasoning"] = reasoning_match.group(1).strip().strip('"')
                    else:
                        result["reasoning"] = "No reasoning provided"
                return result

            raise ValueError(f"Could not extract valid JSON from response: {text[:200]}...")

    def predict_single(self, question, answer, temperature=0.7):
        """
        Predict CLARITY label for a single question-answer pair.
        
        Args:
            question: Interview question
            answer: Interview answer
            temperature: Sampling temperature
        
        Returns:
            Dict with 'label' and 'reasoning' keys, or None if parsing fails
        """
        prompt = self.build_prompt(question, answer)
        
        messages = [
            {"role": "user", "content": prompt}
        ]
        
        try:
            response = call_granite_model(messages, temperature=temperature)
            parsed = self.extract_json(response)
            
            # Validate label
            valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
            if parsed.get("label") not in valid_labels:
                # Try to normalize the label
                label_lower = parsed.get("label", "").lower()
                if "direct" in label_lower and "reply" in label_lower:
                    parsed["label"] = "Direct Reply"
                elif "direct" in label_lower and ("non" in label_lower or "decline" in label_lower):
                    parsed["label"] = "Direct Non-Reply"
                elif "indirect" in label_lower or "evasive" in label_lower:
                    parsed["label"] = "Indirect"
                else:
                    # Default fallback
                    parsed["label"] = "Indirect"
            
            return parsed
        except Exception as e:
            print(f"Error in predict_single: {e}")
            return None

    def predict_batch(self, examples):
        """
        Predict CLARITY labels for a batch of examples.
        
        Args:
            examples: List of dicts with 'question' and 'answer' keys
        
        Returns:
            List of dicts with 'label' and 'reasoning' keys
        """
        results = []
        for example in examples:
            question = example.get("question", "")
            answer = example.get("answer", "")
            
            if not question or not answer:
                results.append({"label": "Indirect", "reasoning": "Missing question or answer"})
                continue
            
            prediction = self.predict_single(question, answer, temperature=0.0)
            if prediction:
                results.append(prediction)
            else:
                results.append({"label": "Indirect", "reasoning": "Prediction failed"})
        
        return results

# Import our custom modules
"""
Granite Self-Consistency Strategy for CLARITY

Implements self-consistency by sampling multiple predictions and voting on labels.
Uses GraniteClarityStrategy as the base strategy.
"""

from collections import Counter


class GraniteSelfConsistencyStrategy(GraniteClarityStrategy):
    """Self-consistency wrapper for Granite CLARITY classification."""

    name = "granite-self-consistency"

    def __init__(self, samples=3, temperature=0.7):
        """
        Initialize self-consistency strategy.
        
        Args:
            samples: Number of samples to generate for voting (default: 5)
            temperature: Sampling temperature for diversity (default: 0.7)
        """
        super().__init__()
        self.samples = samples
        self.temperature = temperature

    def predict_single_with_voting(self, question, answer):
        """
        Predict label using self-consistency voting.
        
        Args:
            question: Interview question
            answer: Interview answer
        
        Returns:
            Dict with:
                - 'label': Final voted label
                - 'reasoning': All reasoning traces (list)
                - 'votes': Vote counts for each label
                - 'all_predictions': All individual predictions
        """
        predictions = []
        reasonings = []
        
        # Micro-batching: process 2 samples at a time (safe for 15GB T4 GPU)
        batch_size = 2
        prompt = self.build_prompt(question, answer)
        
        for batch_start in range(0, self.samples, batch_size):
            batch_end = min(batch_start + batch_size, self.samples)
            batch_messages = [
                [{"role": "user", "content": prompt}]
                for _ in range(batch_end - batch_start)
            ]
            
            try:
                # Generate batch
                batch_responses = call_granite_model_batch(
                    batch_messages, 
                    temperature=self.temperature
                )
                
                # Parse each response
                for response in batch_responses:
                    try:
                        parsed = self.extract_json(response)
                        # Validate and normalize label
                        valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
                        if parsed.get("label") not in valid_labels:
                            label_lower = parsed.get("label", "").lower()
                            if "direct" in label_lower and "reply" in label_lower:
                                parsed["label"] = "Direct Reply"
                            elif "direct" in label_lower and ("non" in label_lower or "decline" in label_lower):
                                parsed["label"] = "Direct Non-Reply"
                            elif "indirect" in label_lower or "evasive" in label_lower:
                                parsed["label"] = "Indirect"
                            else:
                                parsed["label"] = "Indirect"
                        
                        predictions.append(parsed["label"])
                        reasonings.append(parsed.get("reasoning", "No reasoning"))
                    except Exception as e:
                        predictions.append("Indirect")
                        reasonings.append(f"Parse error: {str(e)}")
            except Exception as e:
                # Fallback to sequential if batch fails (memory issue or other error)
                print(f"Batch generation failed, falling back to sequential: {e}")
                for i in range(batch_start, batch_end):
                    try:
                        pred = self.predict_single(question, answer, temperature=self.temperature)
                        if pred and pred.get("label"):
                            predictions.append(pred["label"])
                            reasonings.append(pred.get("reasoning", "No reasoning"))
                        else:
                            predictions.append("Indirect")
                            reasonings.append("Prediction failed")
                    except Exception as e2:
                        predictions.append("Indirect")
                        reasonings.append(f"Error: {str(e2)}")
        
        # Count votes
        vote_counts = Counter(predictions)
        most_common = vote_counts.most_common(1)
        
        if most_common:
            final_label = most_common[0][0]
        else:
            final_label = "Indirect"
        
        return {
            "label": final_label,
            "reasoning": reasonings,
            "votes": dict(vote_counts),
            "all_predictions": predictions
        }

    def predict_batch(self, examples):
        """
        Predict labels for a batch using self-consistency.
        
        Args:
            examples: List of dicts with 'question' and 'answer' keys
        
        Returns:
            List of dicts with 'label', 'reasoning', 'votes', 'all_predictions'
        """
        results = []
        
        for idx, example in enumerate(examples):
            question = example.get("question", "")
            answer = example.get("answer", "")
            
            if not question or not answer:
                results.append({
                    "label": "Indirect",
                    "reasoning": ["Missing question or answer"],
                    "votes": {"Indirect": self.samples},
                    "all_predictions": ["Indirect"] * self.samples
                })
                continue
            
            print(f"Processing example {idx+1}/{len(examples)}: {question[:50]}...")
            result = self.predict_single_with_voting(question, answer)
            results.append(result)
        
        return results
"""
Balanced Dataset Loader for CLARITY

Provides utilities for loading balanced test/validation datasets
with equal representation across all three CLARITY labels.
"""

import random
from collections import Counter
from typing import List, Dict, Tuple, Optional
import pandas as pd
from datasets import load_dataset


def stratified_sample(data: List[Dict], label_key: str, samples_per_label: Optional[int] = None) -> List[Dict]:
    """
    Sample data with equal representation per label.
    
    Args:
        data: List of dicts with label_key
        label_key: Key to extract label from each dict
        samples_per_label: Number of samples per label (None = use minimum count)
    
    Returns:
        Balanced list of samples
    """
    # Group by label
    label_groups = {}
    for item in data:
        label = item.get(label_key)
        if label is None:
            continue
        if label not in label_groups:
            label_groups[label] = []
        label_groups[label].append(item)
    
    # Determine samples per label
    if samples_per_label is None:
        # Use minimum count across all labels
        counts = [len(items) for items in label_groups.values()]
        if not counts:
            return []
        samples_per_label = min(counts)
    
    # Sample equally from each label
    balanced_data = []
    for label, items in label_groups.items():
        # Sample without replacement
        sampled = random.sample(items, min(samples_per_label, len(items)))
        balanced_data.extend(sampled)
    
    return balanced_data


def load_balanced_test_data(
    split: str = "test",
    samples_per_label: Optional[int] = None,
    dataset_name: str = "ailsntua/QEvasion"
) -> Tuple[List[Dict], List[str]]:
    """
    Load balanced test data from QEvasion dataset.
    
    Args:
        split: Dataset split to use ("test" or "train")
        samples_per_label: Number of samples per label (None = use minimum)
        dataset_name: HuggingFace dataset name
    
    Returns:
        Tuple of (examples, labels) where examples are dicts with 'question' and 'answer'
    """
    print(f"Loading {split} split from {dataset_name}...")
    dataset = load_dataset(dataset_name)
    
    if split not in dataset:
        raise ValueError(f"Split '{split}' not found in dataset. Available: {list(dataset.keys())}")
    
    split_data = dataset[split]
    
    # Convert to list of dicts
    examples = []
    for item in split_data:
        # Map clarity_label to CLARITY format
        clarity_label = item.get("clarity_label", "")
        
        # Map QEvasion labels to CLARITY format
        label_mapping = {
            "Clear Reply": "Direct Reply",
            "Clear Non-Reply": "Direct Non-Reply",
            "Ambivalent Reply": "Indirect",
            "Ambivalent": "Indirect",
        }
        
        mapped_label = label_mapping.get(clarity_label, clarity_label)
        
        # Only include valid CLARITY labels
        if mapped_label not in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
            continue
        
        examples.append({
            "question": str(item.get("interview_question", item.get("question", ""))),
            "answer": str(item.get("interview_answer", "")),
            "clarity_label": mapped_label,
            "original_label": clarity_label
        })
    
    print(f"Loaded {len(examples)} examples from {split} split")
    
    # Show label distribution before balancing
    labels_before = [ex["clarity_label"] for ex in examples]
    label_counts = Counter(labels_before)
    print(f"Label distribution before balancing: {dict(label_counts)}")
    
    # Balance the dataset
    balanced_examples = stratified_sample(examples, "clarity_label", samples_per_label)
    
    # Show label distribution after balancing
    labels_after = [ex["clarity_label"] for ex in balanced_examples]
    label_counts_after = Counter(labels_after)
    print(f"Label distribution after balancing: {dict(label_counts_after)}")
    print(f"Total balanced samples: {len(balanced_examples)}")
    
    # Extract labels
    labels = [ex["clarity_label"] for ex in balanced_examples]
    
    return balanced_examples, labels


def load_clarity_eval_data(
    eval_file: str = "/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv"
) -> Tuple[List[Dict], List[int]]:
    """
    Load CLARITY evaluation dataset (no labels).
    
    Args:
        eval_file: Path to evaluation CSV file
    
    Returns:
        Tuple of (examples, indices) where examples are dicts with 'question' and 'answer'
    """
    print(f"Loading CLARITY evaluation dataset from {eval_file}...")
    
    try:
        df = pd.read_csv(eval_file)
    except FileNotFoundError:
        raise FileNotFoundError(f"Evaluation file not found: {eval_file}")
    
    examples = []
    indices = []
    
    for idx, row in df.iterrows():
        question = str(row.get("interview_question", row.get("question", ""))).strip()
        answer = str(row.get("interview_answer", "")).strip()
        
        if question and answer:
            examples.append({
                "question": question,
                "answer": answer,
                "index": idx
            })
            indices.append(idx)
    
    print(f"Loaded {len(examples)} examples from evaluation dataset")
    
    return examples, indices


def get_label_distribution(data: List[Dict], label_key: str) -> Dict[str, int]:
    """Get label distribution from data."""
    labels = [item.get(label_key) for item in data if item.get(label_key)]
    return dict(Counter(labels))

## Configuration

Set parameters for evaluation:

In [ ]:
def load_eval_from_csv(
    csv_path: str,
    samples_per_label: Optional[int] = None,
) -> Tuple[List[Dict], List[str]]:
    """
    Load eval/test data from a QEvasion-style CSV (e.g. dataset/qevasion_test_308.csv).
    Same format as load_balanced_test_data. Set EVAL_INPUT_CSV in config to use this.
    """
    df = pd.read_csv(csv_path)
    q_col = "question" if "question" in df.columns else "interview_question"
    a_col = "interview_answer" if "interview_answer" in df.columns else "answer"
    lbl_col = "clarity_label"
    if q_col not in df.columns or a_col not in df.columns or lbl_col not in df.columns:
        raise ValueError(f"CSV must have {q_col}/{a_col}/{lbl_col}")
    label_mapping = {
        "Clear Reply": "Direct Reply",
        "Clear Non-Reply": "Direct Non-Reply",
        "Ambivalent Reply": "Indirect",
        "Ambivalent": "Indirect",
    }
    examples = []
    for _, row in df.iterrows():
        clarity_label = str(row.get(lbl_col, "")).strip()
        mapped = label_mapping.get(clarity_label, clarity_label)
        if mapped not in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
            continue
        examples.append({
            "question": str(row.get(q_col, "")),
            "answer": str(row.get(a_col, "")),
            "clarity_label": mapped,
            "original_label": clarity_label,
        })
    labels = [ex["clarity_label"] for ex in examples]
    if samples_per_label is not None:
        examples = stratified_sample(examples, "clarity_label", samples_per_label)
        labels = [ex["clarity_label"] for ex in examples]
    print(f"Loaded {len(examples)} examples from CSV: {csv_path}")
    return examples, labels

In [3]:
# Configuration
SAMPLES = 3  # Number of self-consistency samples
TEMPERATURE = 0.7  # Sampling temperature
TEST_SAMPLES_PER_LABEL = None  # None = use minimum count across labels
MAX_TEST_SAMPLES = None  # None = evaluate all balanced samples
EVAL_FILE = "/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv"
# Optional: path to saved QEvasion test CSV. If empty, uses dataset/qevasion_test_308.csv when present (from clear-non-reply-evaluation-roberta.ipynb); else HuggingFace.
EVAL_INPUT_CSV = ""
DEFAULT_EVAL_CSV = "dataset/qevasion_test_308.csv"
OUTPUT_FILE = "clarity_submission_granite.pickle"

print(f"Configuration:")
print(f"  Self-consistency samples: {SAMPLES}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Test samples per label: {TEST_SAMPLES_PER_LABEL}")
print(f"  Max test samples: {MAX_TEST_SAMPLES}")

Configuration:
  Self-consistency samples: 3
  Temperature: 0.7
  Test samples per label: None
  Max test samples: None


## Initialize Strategy

## Training with PEFT/LoRA

Fine-tune Granite 3.2-2B-Instruct on rationale dataset with PEFT/LoRA for CLARITY classification.

**Note:** Set `TRAIN_MODEL = True` below to enable training. If `False`, the notebook will skip training and use the base model for evaluation.

### 📚 Google Drive Setup (Colab) - Quick Start

**For Colab users:**

1. **Run the Drive mount cell** (Cell 7) - it will prompt you to authorize Google Drive access
2. **Upload your rationale CSV** to `/content/drive/MyDrive/granite_clarity/data/`
   - Click the folder icon in Colab sidebar → Upload files
   - Or use: `from google.colab import files; files.upload()`
3. **Update `RATIONALE_CSV_PATH`** in the config cell to match your uploaded filename
4. **Set `TRAIN_MODEL = True`** to start training
5. **Models are automatically saved** to `/content/drive/MyDrive/granite_clarity/models/` - they persist after Colab session ends!

**To load a saved model later:**
- Set `LOAD_SAVED_MODEL = True` and `TRAIN_MODEL = False` in the cells below
- Update `SAVED_MODEL_PATH` to point to your saved model directory

### ☁️ Load CSV from Cloud

You can load the CSV file directly from cloud storage:
- **Google Drive share link**: Set `CSV_CLOUD_URL` to a Google Drive share link
- **Direct URL**: Set `CSV_CLOUD_URL` to any HTTP/HTTPS URL pointing to a CSV file
- **Auto-detect**: Leave `CSV_CLOUD_URL` empty to auto-detect CSV files in Drive or use local paths

In [4]:
# =============================================================================
# Google Drive Setup (Colab) - Run this cell first on Colab!
# =============================================================================
import os
from pathlib import Path
import glob

# Detect if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running on Google Colab")
    
    # Mount Google Drive (will prompt for authorization)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set base directories
    DRIVE_BASE = Path("/content/drive/MyDrive")
    COLAB_BASE = Path("/content")
    
    # Create directories if they don't exist
    (DRIVE_BASE / "granite_clarity").mkdir(exist_ok=True, parents=True)
    (DRIVE_BASE / "granite_clarity" / "models").mkdir(exist_ok=True, parents=True)
    (DRIVE_BASE / "granite_clarity" / "data").mkdir(exist_ok=True, parents=True)
    
    print(f"✅ Google Drive mounted at {DRIVE_BASE}")
    print(f"   Models will be saved to: {DRIVE_BASE / 'granite_clarity' / 'models'}")
    print(f"   Data directory: {DRIVE_BASE / 'granite_clarity' / 'data'}")
    
    # Auto-detect CSV files in the data directory
    data_dir = DRIVE_BASE / "granite_clarity" / "data"
    csv_files = list(data_dir.glob("*.csv"))
    if csv_files:
        print(f"\n📁 Found {len(csv_files)} CSV file(s) in data directory:")
        for csv_file in csv_files:
            size_mb = csv_file.stat().st_size / (1024 * 1024)
            print(f"   - {csv_file.name} ({size_mb:.2f} MB)")
    else:
        print(f"\n⚠️  No CSV files found in {data_dir}")
        print(f"   Run the upload cell below to upload your CSV file")
    
except ImportError:
    IN_COLAB = False
    print("⚠️  Not running on Colab - using local paths")
    DRIVE_BASE = Path("/Users/andrearachetta/Desktop")
    COLAB_BASE = Path("/Users/andrearachetta/Desktop")

# =============================================================================
# Training Configuration
# =============================================================================
TRAIN_MODEL = True  # Set to True to enable training, False to skip training

# =============================================================================
# CUDA Debugging Configuration
# =============================================================================
# Enable CUDA debugging for better error reporting
# Set to True to get synchronous CUDA errors (slower but more accurate)
ENABLE_CUDA_DEBUG = True  # Set to True if you're getting CUDA errors

# Set CUDA_LAUNCH_BLOCKING at the very beginning (before any CUDA operations)
import os
if ENABLE_CUDA_DEBUG:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    print("⚠️  CUDA_LAUNCH_BLOCKING enabled - training will be slower but errors will be more accurate")
    print("   This helps identify the exact location of CUDA errors")
else:
    # Make sure it's not set if disabled
    os.environ.pop("CUDA_LAUNCH_BLOCKING", None)

# =============================================================================
# Cloud CSV Configuration (Optional)
# =============================================================================
# Set CSV_CLOUD_URL to load CSV from cloud instead of local file
# Examples:
#   - Google Drive share link: "https://drive.google.com/file/d/FILE_ID/view?usp=sharing"
#   - Direct URL: "https://example.com/data.csv"
#   - Leave empty to use local/Drive file
CSV_CLOUD_URL = ""  # Set this to load from cloud, or leave empty for local/Drive

# Paths (automatically adjust for Colab vs local)
if IN_COLAB:
    # Colab: Use Google Drive for persistence
    # Auto-detect CSV file in data directory
    data_dir = DRIVE_BASE / "granite_clarity" / "data"
    csv_files = list(data_dir.glob("*.csv"))
    
    if csv_files:
        # Use the newest/largest CSV file
        csv_files.sort(key=lambda x: (x.stat().st_mtime, x.stat().st_size), reverse=True)
        RATIONALE_CSV_PATH = str(csv_files[0])
        print(f"\n✅ Auto-detected CSV: {csv_files[0].name}")
    else:
        # Default path if no CSV found
        RATIONALE_CSV_PATH = str(DRIVE_BASE / "granite_clarity" / "data" / "qevasion_rationale_dataset.csv")
        print(f"\n⚠️  No CSV found in data directory - using default path")
        print(f"   Upload CSV using the upload cell above, or update RATIONALE_CSV_PATH manually")
    
    TRAINING_OUTPUT_DIR = str(DRIVE_BASE / "granite_clarity" / "models" / "granite_clarity_finetuned")
    
    # Use cloud URL if provided, otherwise use local/Drive path
    if CSV_CLOUD_URL:
        RATIONALE_CSV_PATH = CSV_CLOUD_URL
        print(f"\n🌐 Using cloud CSV source: {CSV_CLOUD_URL[:80]}...")
    else:
        print(f"\n📁 Colab paths:")
        print(f"   Rationale CSV: {RATIONALE_CSV_PATH}")
    
    print(f"   Model output: {TRAINING_OUTPUT_DIR}")
else:
    # Local paths (Mac/Linux) - try to find CSV automatically
    possible_csv_paths = [
        "/Users/andrearachetta/Desktop/qevasion_rationale/qevasion_rationale_dataset_20260204_163024.csv",
        "/Users/andrearachetta/Desktop/qevasion_rationale/*.csv",
    ]
    
    RATIONALE_CSV_PATH = None
    for pattern in possible_csv_paths:
        matches = glob.glob(pattern)
        if matches:
            RATIONALE_CSV_PATH = matches[0]
            break
    
    if not RATIONALE_CSV_PATH or not os.path.exists(RATIONALE_CSV_PATH):
        # Default fallback
        RATIONALE_CSV_PATH = "/Users/andrearachetta/Desktop/qevasion_rationale/qevasion_rationale_dataset_20260204_163024.csv"
        print(f"\n⚠️  CSV not found at default location")
    
    TRAINING_OUTPUT_DIR = "/Users/andrearachetta/Desktop/granite_clarity_finetuned"
    print(f"\n📁 Local paths:")
    print(f"   Rationale CSV: {RATIONALE_CSV_PATH}")
    print(f"   Model output: {TRAINING_OUTPUT_DIR}")

# Training hyperparameters
TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 2
TRAIN_LEARNING_RATE = 2e-5
TRAIN_MAX_LENGTH = 1024
TRAIN_GRADIENT_ACCUMULATION_STEPS = 2
TRAIN_SAVE_STEPS = 100
TRAIN_EVAL_STEPS = 50
TRAIN_MAX_TRAIN_SAMPLES = None  # None = use all available

# PEFT/LoRA configuration
USE_PEFT = True  # Always use PEFT (recommended)
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Few-shot configuration (for training prompts)
TRAIN_FEW_SHOT = 0  # Number of few-shot examples in training prompts (0 = no few-shot)

print(f"Training configuration:")
print(f"  Train model: {TRAIN_MODEL}")
print(f"  Epochs: {TRAIN_EPOCHS}")
print(f"  Batch size: {TRAIN_BATCH_SIZE}")
print(f"  Learning rate: {TRAIN_LEARNING_RATE}")
print(f"  Use PEFT: {USE_PEFT}")
if USE_PEFT:
    print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

✅ Running on Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive/MyDrive
   Models will be saved to: /content/drive/MyDrive/granite_clarity/models
   Data directory: /content/drive/MyDrive/granite_clarity/data

📁 Found 1 CSV file(s) in data directory:
   - qevasion_rationale_dataset_20260204_163024.csv (0.92 MB)
⚠️  CUDA_LAUNCH_BLOCKING enabled - training will be slower but errors will be more accurate
   This helps identify the exact location of CUDA errors

✅ Auto-detected CSV: qevasion_rationale_dataset_20260204_163024.csv

📁 Colab paths:
   Rationale CSV: /content/drive/MyDrive/granite_clarity/data/qevasion_rationale_dataset_20260204_163024.csv
   Model output: /content/drive/MyDrive/granite_clarity/models/granite_clarity_finetuned
Training configuration:
  Train model: True
  Epochs: 2
  Batch size: 2
  Learning rate: 2e-05
  Use PEFT: True
  LoRA r=8,

### 📤 Upload CSV File to Colab

**Option 1: Upload via Colab file picker (recommended)**
Run the cell below to upload your CSV file directly.

In [5]:
# =============================================================================
# Upload CSV File to Google Drive (Colab)
# =============================================================================
# Run this cell to upload your rationale CSV file to Google Drive
# Automatically skips if CSV files already exist in Drive

if IN_COLAB:
    from google.colab import files
    import shutil
    
    print("=" * 60)
    print("UPLOAD CSV FILE")
    print("=" * 60)
    
    # Check if CSV files already exist in Drive
    data_dir = DRIVE_BASE / "granite_clarity" / "data"
    data_dir.mkdir(exist_ok=True, parents=True)
    existing_csv_files = list(data_dir.glob("*.csv"))
    
    if existing_csv_files:
        print(f"✅ CSV file(s) already found in Drive:")
        for csv_file in existing_csv_files:
            size_mb = csv_file.stat().st_size / (1024 * 1024)
            print(f"   - {csv_file.name} ({size_mb:.2f} MB)")
        print(f"\n⏭️  Skipping upload - files already exist in Drive")
        print(f"   Location: {data_dir}")
        print(f"\n💡 To upload a new file, delete existing files first or use a different filename")
    else:
        print("No CSV files found in Drive data directory.")
        print("Click 'Choose Files' below to upload your rationale CSV file.")
        print("The file will be saved to Google Drive for persistence.")
        print()
        
        # Upload file
        uploaded = files.upload()
        
        if uploaded:
            # Move uploaded file to Drive data directory
            for filename, file_content in uploaded.items():
                dest_path = data_dir / filename
                with open(dest_path, 'wb') as f:
                    f.write(file_content)
                size_mb = len(file_content) / (1024 * 1024)
                print(f"✅ Uploaded: {filename} ({size_mb:.2f} MB)")
                print(f"   Saved to: {dest_path}")
            
            print(f"\n📁 Files in data directory:")
            csv_files = list(data_dir.glob("*.csv"))
            for csv_file in csv_files:
                print(f"   - {csv_file.name}")
            
            if csv_files:
                print(f"\n💡 CSV will be auto-detected in the config cell")
        else:
            print("⚠️  No files uploaded")
else:
    print("⚠️  Not running on Colab - upload cell skipped")
    print("   On local machine, make sure your CSV is at the path specified in RATIONALE_CSV_PATH")

UPLOAD CSV FILE
✅ CSV file(s) already found in Drive:
   - qevasion_rationale_dataset_20260204_163024.csv (0.92 MB)

⏭️  Skipping upload - files already exist in Drive
   Location: /content/drive/MyDrive/granite_clarity/data

💡 To upload a new file, delete existing files first or use a different filename


### ☁️ Load CSV from Cloud URL

**Option 1: Google Drive Share Link**
1. Upload your CSV to Google Drive
2. Right-click → "Get link" → Set to "Anyone with the link"
3. Copy the share link (e.g., `https://drive.google.com/file/d/FILE_ID/view?usp=sharing`)
4. Set `CSV_CLOUD_URL` in the config cell above to this link

**Option 2: Direct HTTP/HTTPS URL**
- Set `CSV_CLOUD_URL` to any URL pointing to a CSV file
- Example: `CSV_CLOUD_URL = "https://example.com/data.csv"`

**Option 3: Auto-detect from Drive**
- Leave `CSV_CLOUD_URL` empty and upload CSV to Drive's data directory
- The code will auto-detect CSV files in `/content/drive/MyDrive/granite_clarity/data/`

### 🔍 Auto-detect CSV File

Run this cell to automatically find and use CSV files in your Drive data directory:

In [6]:
# =============================================================================
# Auto-detect CSV File in Drive
# =============================================================================
# This cell will automatically find CSV files and set RATIONALE_CSV_PATH

if IN_COLAB:
    data_dir = DRIVE_BASE / "granite_clarity" / "data"
    csv_files = list(data_dir.glob("*.csv"))
    
    if csv_files:
        # Sort by modification time (newest first) and file size (largest first)
        csv_files.sort(key=lambda x: (x.stat().st_mtime, x.stat().st_size), reverse=True)
        
        print("=" * 60)
        print("AUTO-DETECTED CSV FILES")
        print("=" * 60)
        print(f"Found {len(csv_files)} CSV file(s):\n")
        
        for i, csv_file in enumerate(csv_files, 1):
            size_mb = csv_file.stat().st_size / (1024 * 1024)
            mod_time = os.path.getmtime(csv_file)
            from datetime import datetime
            mod_str = datetime.fromtimestamp(mod_time).strftime("%Y-%m-%d %H:%M:%S")
            print(f"{i}. {csv_file.name}")
            print(f"   Size: {size_mb:.2f} MB")
            print(f"   Modified: {mod_str}")
            print(f"   Path: {csv_file}")
            print()
        
        # Use the first (newest/largest) file
        selected_csv = csv_files[0]
        print(f"✅ Auto-selected: {selected_csv.name}")
        print(f"   Update RATIONALE_CSV_PATH = '{selected_csv}' in the config cell")
        
        # Optionally auto-update the variable (uncomment to enable)
        # RATIONALE_CSV_PATH = str(selected_csv)
        # print(f"   ✅ RATIONALE_CSV_PATH automatically updated!")
    else:
        print("⚠️  No CSV files found in data directory")
        print(f"   Expected location: {data_dir}")
        print(f"   Please upload a CSV file using the upload cell above")
else:
    # Local: try to find CSV in common locations
    possible_paths = [
        "/Users/andrearachetta/Desktop/qevasion_rationale/qevasion_rationale_dataset_20260204_163024.csv",
        "/Users/andrearachetta/Desktop/qevasion_rationale/*.csv",
        str(DRIVE_BASE / "qevasion_rationale" / "*.csv"),
    ]
    
    print("=" * 60)
    print("SEARCHING FOR CSV FILES (Local)")
    print("=" * 60)
    
    found_files = []
    for pattern in possible_paths:
        matches = glob.glob(pattern)
        found_files.extend(matches)
    
    if found_files:
        print(f"Found {len(found_files)} CSV file(s):\n")
        for i, csv_file in enumerate(found_files, 1):
            if os.path.exists(csv_file):
                size_mb = os.path.getsize(csv_file) / (1024 * 1024)
                print(f"{i}. {csv_file} ({size_mb:.2f} MB)")
        print(f"\n💡 Using: {found_files[0]}")
    else:
        print("⚠️  No CSV files found in common locations")
        print("   Make sure RATIONALE_CSV_PATH points to your CSV file")

AUTO-DETECTED CSV FILES
Found 1 CSV file(s):

1. qevasion_rationale_dataset_20260204_163024.csv
   Size: 0.92 MB
   Modified: 2026-02-04 17:10:05
   Path: /content/drive/MyDrive/granite_clarity/data/qevasion_rationale_dataset_20260204_163024.csv

✅ Auto-selected: qevasion_rationale_dataset_20260204_163024.csv
   Update RATIONALE_CSV_PATH = '/content/drive/MyDrive/granite_clarity/data/qevasion_rationale_dataset_20260204_163024.csv' in the config cell


### 🔧 CUDA Error Recovery

If you're getting CUDA errors, run this cell first to reset CUDA state:

In [7]:
# =============================================================================
# CUDA Error Recovery - Run this if you're getting CUDA errors
# =============================================================================
# This cell resets CUDA state and clears any lingering errors

import torch
import gc

print("=" * 60)
print("CUDA ERROR RECOVERY")
print("=" * 60)

if torch.cuda.is_available():
    print("Resetting CUDA state...")
    
    # Clear any Python references
    gc.collect()
    
    # Try to reset CUDA error state
    try:
        # Check for existing errors
        torch.cuda.synchronize()
        print("✅ CUDA synchronized successfully")
    except RuntimeError as e:
        print(f"⚠️  CUDA error detected: {e}")
        print("   Attempting to reset...")
        
        # Try to clear error state
        try:
            # Force reset by clearing cache
            torch.cuda.empty_cache()
            print("✅ CUDA cache cleared")
        except:
            pass
        
        # Try to get device count (this sometimes resets state)
        try:
            device_count = torch.cuda.device_count()
            print(f"✅ CUDA devices detected: {device_count}")
        except:
            print("⚠️  Could not query CUDA devices")
    
    # Final cleanup
    gc.collect()
    try:
        torch.cuda.empty_cache()
        print("✅ Final CUDA cache clear completed")
    except RuntimeError as e:
        print(f"❌ CUDA still in error state: {e}")
        print("\n💡 RECOMMENDED ACTIONS:")
        print("   1. Restart the kernel (Runtime → Restart runtime)")
        print("   2. Set ENABLE_CUDA_DEBUG = True in config cell")
        print("   3. Reduce batch size: TRAIN_BATCH_SIZE = 1")
        print("   4. Try CPU training if GPU issues persist")
else:
    print("⚠️  CUDA not available - skipping CUDA reset")

print("\n" + "=" * 60)
print("If errors persist, restart the kernel and try again")
print("=" * 60)

CUDA ERROR RECOVERY
Resetting CUDA state...
✅ CUDA synchronized successfully
✅ Final CUDA cache clear completed

If errors persist, restart the kernel and try again


In [8]:
# =============================================================================
# Training Functions (from train_granite_rationale.py)
# =============================================================================
import gc
import json
import random
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any
import pandas as pd
import torch
# Force torch._dynamo to fully load before transformers (avoids AttributeError when
# transformers loads flex_attention and uses torch.compiler.disable)
try:
    import torch._dynamo.guards
except Exception:
    pass
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Label mapping: QEvasion -> CLARITY submission format
CLARITY_LABEL_MAP = {
    "Clear Reply": "Direct Reply",
    "Clear Non-Reply": "Direct Non-Reply",
    "Ambivalent Reply": "Indirect",
    "Ambivalent": "Indirect",
}

def load_csv_from_cloud(url_or_path: str, cache_dir: Optional[str] = None) -> str:
    """
    Load CSV from cloud source (URL, Google Drive share link, etc.).
    Downloads and caches the file locally.
    
    Args:
        url_or_path: URL, Google Drive share link, or local file path
        cache_dir: Directory to cache downloaded files (default: Drive data dir or /tmp)
    
    Returns:
        Local file path to the CSV
    """
    import urllib.request
    import re
    
    # If it's already a local file, return as-is
    if os.path.exists(url_or_path):
        return url_or_path
    
    # Setup cache directory
    if cache_dir is None:
        try:
            if IN_COLAB:
                cache_dir = str(DRIVE_BASE / "granite_clarity" / "data")
            else:
                cache_dir = "/tmp/granite_clarity_cache"
        except NameError:
            cache_dir = "/tmp/granite_clarity_cache"
    Path(cache_dir).mkdir(exist_ok=True, parents=True)
    
    # Handle Google Drive share links
    if "drive.google.com" in url_or_path:
        # Extract file ID from Google Drive share link
        # Format: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
        # Or: https://drive.google.com/open?id=FILE_ID
        file_id_match = re.search(r'/d/([a-zA-Z0-9_-]+)', url_or_path) or re.search(r'id=([a-zA-Z0-9_-]+)', url_or_path)
        if file_id_match:
            file_id = file_id_match.group(1)
            # Use direct download link
            download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
            filename = f"gdrive_{file_id}.csv"
        else:
            raise ValueError(f"Could not extract file ID from Google Drive URL: {url_or_path}")
    else:
        # Regular URL
        download_url = url_or_path
        filename = os.path.basename(url_or_path) or "downloaded.csv"
    
    cache_path = Path(cache_dir) / filename
    
    # Download if not cached
    if not cache_path.exists():
        print(f"📥 Downloading CSV from cloud: {url_or_path[:80]}...")
        try:
            urllib.request.urlretrieve(download_url, cache_path)
            size_mb = cache_path.stat().st_size / (1024 * 1024)
            print(f"✅ Downloaded: {filename} ({size_mb:.2f} MB) -> {cache_path}")
        except Exception as e:
            raise RuntimeError(f"Failed to download CSV from {url_or_path}: {e}") from e
    else:
        print(f"✅ Using cached CSV: {cache_path}")
    
    return str(cache_path)

def load_rationale_csv(csv_path: str, use_cloud: bool = True) -> pd.DataFrame:
    """
    Load rationale CSV from local path or cloud source.
    
    Args:
        csv_path: Local file path, URL, or Google Drive share link
        use_cloud: If True, try to load from cloud if local file not found
    
    Returns:
        DataFrame with filtered rationale data
    """
    # Try local path first
    if not os.path.exists(csv_path) and use_cloud:
        # Check if it looks like a URL or cloud path
        if csv_path.startswith(("http://", "https://", "gs://", "s3://")) or "drive.google.com" in csv_path:
            print(f"🌐 Loading CSV from cloud source...")
            csv_path = load_csv_from_cloud(csv_path)
        else:
            # Try to find in Drive if on Colab
            try:
                if IN_COLAB:
                    drive_path = DRIVE_BASE / "granite_clarity" / "data" / os.path.basename(csv_path)
                    if drive_path.exists():
                        print(f"📁 Found CSV in Drive: {drive_path}")
                        csv_path = str(drive_path)
                    else:
                        raise FileNotFoundError(
                            f"CSV not found locally and not a cloud URL: {csv_path}\n"
                            f"   Upload CSV to Drive or provide a cloud URL (Google Drive share link, HTTP URL, etc.)"
                        )
                else:
                    raise FileNotFoundError(f"CSV file not found: {csv_path}")
            except NameError:
                raise FileNotFoundError(f"CSV file not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    df["verdict_match"] = df["verdict_match"].astype(str).str.lower().eq("true")
    df = df[df["verdict_match"]]
    df = df[df["initial_reasoning"].notna() & (df["initial_reasoning"].astype(str).str.len() > 0)]
    df["final_verdict"] = df["final_verdict"].fillna(df["initial_verdict"]).fillna(df["clarity_label"])
    return df

def row_to_clarity_label(row: pd.Series) -> str:
    """Map a row's label to CLARITY format."""
    label = row.get("final_verdict") or row.get("clarity_label") or row.get("initial_verdict")
    if pd.isna(label):
        return "Indirect"
    label = str(label).strip()
    return CLARITY_LABEL_MAP.get(label, label if label in ("Direct Reply", "Direct Non-Reply", "Indirect") else "Indirect")

def build_assistant_output(row: pd.Series) -> str:
    """Build target assistant output: JSON with reasoning and label."""
    reasoning = str(row.get("initial_reasoning") or "").strip()
    correction = row.get("correction_applied")
    if correction in (True, "True", "true", "1", 1):
        corrective = str(row.get("corrective_reasoning") or "").strip()
        if corrective:
            reasoning = reasoning + "\n\n[Correction]\n" + corrective
    label = row_to_clarity_label(row)
    return json.dumps({"reasoning": reasoning, "label": label}, ensure_ascii=False)

def build_user_prompt(question: str, answer: str, few_shot_examples: Optional[List[Dict]] = None) -> str:
    """Build user prompt (same format as evaluation)."""
    prompt = f"""You are analyzing political interview answers for clarity classification.

Question: {question}
Answer: {answer}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Provide your reasoning and then classify as one of:
- "Direct Reply": Directly answers the question
- "Direct Non-Reply": Explicitly declines or claims inability to answer
- "Indirect": Evasive, indirect, or partially answers

Respond in JSON format:
{{
  "reasoning": "Your step-by-step analysis...",
  "label": "Direct Reply|Direct Non-Reply|Indirect"
}}"""
    if few_shot_examples:
        examples_text = []
        for ex in few_shot_examples:
            examples_text.append(
                f"Example:\nQuestion: {ex['question'][:200]}...\nAnswer: {ex['answer'][:200]}...\n"
                f"Response: {ex['output']}"
            )
        prompt = "Here are some examples.\n\n" + "\n\n".join(examples_text) + "\n\nNow do the following.\n\n" + prompt
    return prompt

def build_training_examples(
    df: pd.DataFrame,
    tokenizer,
    max_length: int = 1024,
    few_shot_pool: Optional[pd.DataFrame] = None,
    num_few_shot: int = 0,
) -> List[Dict]:
    """Build training examples from rationale DataFrame."""
    examples = []
    few_shot_list = None
    
    if num_few_shot > 0 and few_shot_pool is not None and len(few_shot_pool) >= num_few_shot:
        few_shot_list = []
        sampled = few_shot_pool.sample(n=num_few_shot, random_state=42)
        for _, row in sampled.iterrows():
            few_shot_list.append({
                "question": str(row.get("interview_question", "")),
                "answer": str(row.get("interview_answer", "")),
                "output": build_assistant_output(row),
            })
    
    for _, row in df.iterrows():
        question = str(row.get("interview_question", ""))
        answer = str(row.get("interview_answer", ""))
        if not question or not answer:
            continue
        
        user_prompt = build_user_prompt(question, answer, few_shot_examples=few_shot_list)
        assistant_output = build_assistant_output(row)
        
        messages = [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_output},
        ]
        
        try:
            formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:
            formatted = user_prompt + "\n\n" + assistant_output
        
        tokenized = tokenizer(
            formatted,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors=None,
        )
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        
        # Find where assistant starts (mask loss on user part)
        try:
            prompt_only = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompt_ids = tokenizer(prompt_only, return_tensors=None, add_special_tokens=False)["input_ids"]
            prompt_len = len(prompt_ids)
        except Exception:
            prompt_len = len(tokenizer(user_prompt, return_tensors=None, add_special_tokens=False)["input_ids"])
        
        labels = [-100] * len(input_ids)
        vocab_size = len(tokenizer) if hasattr(tokenizer, "__len__") else tokenizer.vocab_size
        
        # Validate and set labels (only for assistant tokens)
        for i in range(prompt_len, min(len(input_ids), max_length)):
            token_id = input_ids[i]
            # Validate token ID is within vocabulary range
            if 0 <= token_id < vocab_size:
                labels[i] = token_id
            else:
                # Invalid token ID - keep as -100 (ignored in loss)
                labels[i] = -100
        
        # Additional validation: check for any invalid token IDs in input_ids
        invalid_tokens = [idx for idx, tid in enumerate(input_ids) if tid < 0 or tid >= vocab_size]
        if invalid_tokens:
            print(f"⚠️  Warning: Found {len(invalid_tokens)} invalid token IDs in example (will be masked)")
        
        examples.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        })
    
    print(f"✅ Built {len(examples)} training examples (vocab_size={vocab_size})")
    return examples

class EvalVotingCallback(TrainerCallback):
    """Run evaluation with voting at save_steps."""
    
    def __init__(self, eval_examples: List[Dict], tokenizer, num_samples: int = 3):
        self.eval_examples = eval_examples
        self.tokenizer = tokenizer
        self.num_samples = num_samples
    
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step > 0 and args.save_steps > 0 and state.global_step % args.save_steps == 0:
            if model is None:
                return
            acc, macro_f1, _, _ = evaluate_with_voting(
                model, self.tokenizer, self.eval_examples, num_samples=self.num_samples
            )
            print(f"[Eval @ step {state.global_step}] accuracy={acc:.4f}, macro_f1={macro_f1:.4f}")

def evaluate_with_voting(
    model,
    tokenizer,
    eval_examples: List[Dict],
    num_samples: int = 3,
    temperature: float = 0.7,
    max_new_tokens: int = 512,
    few_shot_examples: Optional[List[Dict]] = None,
) -> Tuple[float, float, List[str], List[str]]:
    """Run self-consistency evaluation with voting."""
    from collections import Counter
    
    device = next(model.parameters()).device
    all_preds = []
    all_gold = []
    
    # Use tqdm for progress bar
    eval_iter = tqdm(eval_examples, desc="Evaluating", unit="example", leave=False)
    
    for ex in eval_iter:
        question = ex["question"]
        answer = ex["answer"]
        gold_label = ex["clarity_label"]
        if isinstance(gold_label, str) and gold_label not in ("Direct Reply", "Direct Non-Reply", "Indirect"):
            gold_label = CLARITY_LABEL_MAP.get(gold_label, "Indirect")
        all_gold.append(gold_label)
        
        user_prompt = build_user_prompt(question, answer, few_shot_examples=few_shot_examples)
        messages = [{"role": "user", "content": user_prompt}]
        votes = []
        
        for _ in range(num_samples):
            try:
                formatted = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
                inputs = tokenizer(formatted, return_tensors="pt").to(device)
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                        do_sample=True,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
                m = re.search(r'"label"\s*:\s*"([^"]+)"', text)
                if m:
                    label = m.group(1)
                    if label not in ("Direct Reply", "Direct Non-Reply", "Indirect"):
                        label = CLARITY_LABEL_MAP.get(label, "Indirect")
                    votes.append(label)
                else:
                    votes.append("Indirect")
            except Exception:
                votes.append("Indirect")
        
        majority = Counter(votes).most_common(1)[0][0]
        all_preds.append(majority)
    
    acc = accuracy_score(all_gold, all_preds)
    macro_f1 = f1_score(all_gold, all_preds, average="macro", labels=["Direct Reply", "Direct Non-Reply", "Indirect"])
    return acc, macro_f1, all_preds, all_gold

def load_eval_data_from_qevasion(split: str = "test", max_per_label: Optional[int] = 30) -> List[Dict]:
    """Load balanced eval from QEvasion."""
    from collections import Counter
    ds = load_dataset("ailsntua/QEvasion", split=split)
    examples = []
    for i in range(len(ds)):
        item = ds[i]
        clarity = item.get("clarity_label")
        mapped = CLARITY_LABEL_MAP.get(clarity, clarity)
        if mapped not in ("Direct Reply", "Direct Non-Reply", "Indirect"):
            continue
        examples.append({
            "question": str(item.get("interview_question", item.get("question", ""))),
            "answer": str(item.get("interview_answer", "")),
            "clarity_label": mapped,
        })
    if max_per_label is not None:
        by_label = {}
        for ex in examples:
            L = ex["clarity_label"]
            by_label.setdefault(L, []).append(ex)
        n = min(len(v) for v in by_label.values()) if by_label else 0
        n = min(n, max_per_label)
        examples = []
        for L, lst in by_label.items():
            examples.extend(random.sample(lst, min(n, len(lst))))
        random.shuffle(examples)
    return examples

print("✅ Training functions loaded!")

✅ Training functions loaded!


## Load Saved Model (Optional)

If you've already trained a model and want to load it instead of training again, use this cell:

In [9]:
# =============================================================================
# Load Pre-trained Model from Drive/Local
# =============================================================================
# Set this to True to load a previously saved model instead of training
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend
LOAD_SAVED_MODEL = True
SAVED_MODEL_PATH = TRAINING_OUTPUT_DIR  # Path to saved model directory

if LOAD_SAVED_MODEL:
    print("=" * 60)
    print("LOADING SAVED MODEL")
    print("=" * 60)
    
    if not os.path.exists(SAVED_MODEL_PATH):
        print(f"❌ ERROR: Model not found at {SAVED_MODEL_PATH}")
        if IN_COLAB:
            print(f"   Make sure the model is saved in Google Drive at: {DRIVE_BASE / 'granite_clarity' / 'models'}")
        raise FileNotFoundError(f"Model not found: {SAVED_MODEL_PATH}")
    
    # If the given path has no model weights, look for latest checkpoint (e.g. checkpoints/checkpoint-60)
    model_load_path = os.path.abspath(SAVED_MODEL_PATH)
    has_weights = (
        os.path.isfile(os.path.join(SAVED_MODEL_PATH, "model.safetensors"))
        or os.path.isfile(os.path.join(SAVED_MODEL_PATH, "pytorch_model.bin"))
        or os.path.isfile(os.path.join(SAVED_MODEL_PATH, "adapter_config.json"))
    )
    if not has_weights:
        def _find_checkpoints(parent):
            if not os.path.isdir(parent):
                return []
            return [
                d for d in os.listdir(parent)
                if d.startswith("checkpoint-") and os.path.isdir(os.path.join(parent, d))
            ]
        def _step(name):
            try:
                return int(name.split("-")[1])
            except (IndexError, ValueError):
                return 0
        # 1) Look in SAVED_MODEL_PATH/checkpoints/checkpoint-*
        checkpoint_dir = os.path.join(SAVED_MODEL_PATH, "checkpoints")
        checkpoints = _find_checkpoints(checkpoint_dir)
        # 2) Fallback: look for checkpoint-* directly inside SAVED_MODEL_PATH
        if not checkpoints:
            checkpoints = _find_checkpoints(SAVED_MODEL_PATH)
            checkpoint_dir = SAVED_MODEL_PATH
        if checkpoints:
            latest_name = max(checkpoints, key=_step)
            model_load_path = os.path.abspath(os.path.join(checkpoint_dir, latest_name))
            print(f"Using latest checkpoint: {latest_name}")
        else:
            raise FileNotFoundError(
                f"No model weights or checkpoints found at {SAVED_MODEL_PATH}. "
                f"Looked for model.safetensors/pytorch_model.bin/adapter_config.json, "
                f"and for checkpoints in '{checkpoint_dir}' or directly inside the folder. "
                f"Set SAVED_MODEL_PATH to your checkpoint folder, e.g. .../granite_clarity_finetuned/checkpoints/checkpoint-60"
            )
    
    print(f"Loading model from: {model_load_path}")
    
    # Load tokenizer (from checkpoint or base path; fallback to base if checkpoint has no tokenizer)
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_load_path)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.2-2b-instruct")
        print("   Tokenizer loaded from base model (none in checkpoint)")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Checkpoint tokenizer is sometimes minimal/broken (vocab size 1); use base tokenizer if so
    _vs = getattr(tokenizer, "vocab_size", None)
    if _vs is None and hasattr(tokenizer, "__len__"):
        _vs = len(tokenizer)
    if _vs is not None and _vs < 1000:
        print("   Checkpoint tokenizer has very small vocab. Using base model tokenizer.")
        tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.2-2b-instruct")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
    # Load model
    device = _get_device()
    if device.type == "cuda":
        load_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
    elif device.type == "mps":
        load_kwargs = {"torch_dtype": torch.float32, "device_map": "auto"}
    else:
        load_kwargs = {"torch_dtype": torch.float32}
    
    # Check if it's a PEFT model
    try:
        from peft import PeftModel
        # Load base model without meta device (low_cpu_mem_usage=False) so PEFT adapter loads cleanly
        base_model = AutoModelForCausalLM.from_pretrained(
            "ibm-granite/granite-3.2-2b-instruct",
            **load_kwargs,
            low_cpu_mem_usage=False,
        )
        model = PeftModel.from_pretrained(base_model, model_load_path)
        print("✅ Loaded PEFT/LoRA model")
    except Exception:
        # Fallback to regular model loading
        model = AutoModelForCausalLM.from_pretrained(model_load_path, **load_kwargs)
        print("✅ Loaded full model")
    
    # Update global variables for evaluation
    _granite_model = model
    _granite_tokenizer = tokenizer
    
    print(f"✅ Model loaded successfully from {model_load_path}")
    if TRAIN_MODEL:
        _RESUME_TRAINING_FROM_LOADED = True
        print("   Ready for continued training. Run the Training cell next.")
    else:
        print("   Model is ready for evaluation.")
else:
    print("⏭️  Skipping model loading (LOAD_SAVED_MODEL = False)")
    _RESUME_TRAINING_FROM_LOADED = False

LOADING SAVED MODEL
Using latest checkpoint: checkpoint-60
Loading model from: /content/drive/MyDrive/granite_clarity/models/granite_clarity_finetuned/checkpoint-60
   Checkpoint tokenizer has very small vocab. Using base model tokenizer.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

✅ Loaded PEFT/LoRA model
✅ Model loaded successfully from /content/drive/MyDrive/granite_clarity/models/granite_clarity_finetuned/checkpoint-60
   Ready for continued training. Run the Training cell next.


# Train (model loaded)

In [10]:
# =============================================================================
# Training Execution
# =============================================================================
if TRAIN_MODEL:
    print("=" * 60)
    print("TRAINING GRANITE MODEL WITH PEFT/LoRA")
    print("=" * 60)
    
    # Clear caches and reset CUDA state if needed
    # IMPORTANT: Check CUDA state BEFORE trying to clear cache
    gc.collect()
    
    if torch.cuda.is_available():
        # Check if CUDA is in a good state first (don't call empty_cache if CUDA has errors)
        try:
            # This will raise an error if CUDA is in bad state
            torch.cuda.synchronize()
            # If we get here, CUDA is OK - proceed with cache clear
            torch.cuda.empty_cache()
            print("✅ CUDA cache cleared successfully")
        except RuntimeError as e:
            if "CUDA" in str(e) or "cuda" in str(e).lower() or "device-side assert" in str(e).lower():
                print(f"❌ CUDA ERROR DETECTED: {e}")
                print("\n💡 CUDA is in an error state from a previous operation.")
                print("   Please run the 'CUDA Error Recovery' cell above first,")
                print("   or restart the kernel (Runtime → Restart runtime)")
                raise RuntimeError(
                    "CUDA error detected. Please run the CUDA Error Recovery cell "
                    "or restart the kernel before training."
                ) from e
            else:
                raise
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass
    print("Cleared GPU/MPS/CPU caches.", flush=True)
    
    # Use already-loaded model from checkpoint, or load base model and apply PEFT
    if globals().get("_RESUME_TRAINING_FROM_LOADED", False):
        device = _get_device()
        print("Using model and tokenizer already loaded from checkpoint (skipping reload).")
    else:
        # Load model and tokenizer
        model_name = "ibm-granite/granite-3.2-2b-instruct"
        print(f"Loading model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        # Determine device and dtype
        device = _get_device()
        if device.type == "cuda":
            load_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
        elif device.type == "mps":
            load_kwargs = {"torch_dtype": torch.float32, "device_map": "auto"}
        else:
            load_kwargs = {"torch_dtype": torch.float32}
        
        model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        print(f"Model loaded on {device}")
        
        # Apply PEFT/LoRA (always use PEFT for efficient training)
        if USE_PEFT:
            try:
                from peft import LoraConfig, get_peft_model, TaskType
                lora_config = LoraConfig(
                    r=LORA_R,
                    lora_alpha=LORA_ALPHA,
                    target_modules=LORA_TARGET_MODULES,
                    lora_dropout=LORA_DROPOUT,
                    bias="none",
                    task_type=TaskType.CAUSAL_LM,
                )
                model = get_peft_model(model, lora_config)
                model.print_trainable_parameters()
                print("✅ PEFT/LoRA applied")
            except ImportError:
                print("⚠️  PEFT not installed. Install with: pip install peft")
                USE_PEFT = False
    
    # Enable gradient checkpointing (saves memory)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
        print("✅ Gradient checkpointing enabled")
    
    # Load rationale CSV (supports local, Drive, and cloud sources)
    print(f"\n{'=' * 60}")
    print("LOADING RATIONALE CSV")
    print(f"{'=' * 60}")
    print(f"Source: {RATIONALE_CSV_PATH}")
    
    # Check if it's a cloud URL
    is_cloud_url = (
        RATIONALE_CSV_PATH.startswith(("http://", "https://", "gs://", "s3://")) or 
        "drive.google.com" in RATIONALE_CSV_PATH
    )
    
    if is_cloud_url:
        print(f"🌐 Loading from cloud source...")
    elif not os.path.exists(RATIONALE_CSV_PATH):
        print(f"⚠️  Local file not found, checking cloud/Drive...")
    
    try:
        # load_rationale_csv will handle cloud loading automatically
        df = load_rationale_csv(RATIONALE_CSV_PATH, use_cloud=True)
    except FileNotFoundError as e:
        print(f"❌ ERROR: Could not load CSV")
        print(f"\n📝 Options to fix:")
        if IN_COLAB:
            print(f"   1. Upload CSV to Drive: {DRIVE_BASE / 'granite_clarity' / 'data'}")
            print(f"   2. Set CSV_CLOUD_URL in config cell to a Google Drive share link or URL")
            print(f"   3. Update RATIONALE_CSV_PATH to point to your CSV file")
        else:
            print(f"   1. Set CSV_CLOUD_URL in config cell to a cloud URL")
            print(f"   2. Make sure local CSV exists at: {RATIONALE_CSV_PATH}")
        raise
    print(f"Loaded {len(df)} rationale rows with verdict_match=True")
    
    if TRAIN_MAX_TRAIN_SAMPLES:
        df = df.sample(n=min(TRAIN_MAX_TRAIN_SAMPLES, len(df)), random_state=42)
        print(f"Sampled {len(df)} rows for training")
    
    # Build few-shot pool if needed
    few_shot_pool = None
    if TRAIN_FEW_SHOT > 0:
        few_shot_pool = df.sample(n=min(len(df), max(20, TRAIN_FEW_SHOT * 4)), random_state=43)
        print(f"Few-shot pool: {len(few_shot_pool)} examples")
    
    # Ensure we have a full tokenizer (checkpoint tokenizer can be minimal/broken with vocab size 1)
    _train_vocab_size = getattr(tokenizer, "vocab_size", None)
    if _train_vocab_size is None and hasattr(tokenizer, "__len__"):
        _train_vocab_size = len(tokenizer)
    if _train_vocab_size is None or _train_vocab_size < 1000:
        print("⚠️  Tokenizer has very small vocab (likely from checkpoint). Loading base model tokenizer for training.")
        tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.2-2b-instruct")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
    # Build training dataset
    print("\nBuilding training examples...")
    train_examples = build_training_examples(
        df,
        tokenizer,
        max_length=TRAIN_MAX_LENGTH,
        few_shot_pool=few_shot_pool,
        num_few_shot=TRAIN_FEW_SHOT,
    )
    train_dataset = Dataset.from_list(train_examples)
    print(f"✅ Built {len(train_examples)} training examples")
    
    # Load eval examples for callback
    print("\nLoading eval examples...")
    eval_examples = load_eval_data_from_qevasion(split="test", max_per_label=20)
    print(f"✅ Loaded {len(eval_examples)} eval examples")
    
    # Custom collator
    def _collate_fn(examples):
        batch = {
            "input_ids": torch.tensor([e["input_ids"] for e in examples], dtype=torch.long),
            "attention_mask": torch.tensor([e["attention_mask"] for e in examples], dtype=torch.long),
            "labels": torch.tensor([e["labels"] for e in examples], dtype=torch.long),
        }
        return batch
    
    # Setup checkpoint directory
    checkpoint_dir = Path(TRAINING_OUTPUT_DIR) / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),  # Save checkpoints to checkpoint subdirectory
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=TRAIN_GRADIENT_ACCUMULATION_STEPS,
        learning_rate=TRAIN_LEARNING_RATE,
        logging_steps=10,
        eval_strategy="no",  # We use custom EvalVotingCallback instead
        save_steps=TRAIN_SAVE_STEPS,
        save_total_limit=3,  # Keep multiple checkpoints
        fp16=False,  # Avoid GradScaler "No inf checks" bug with gradient accumulation
        bf16=torch.cuda.is_available(),  # Use bf16 when GPU supports it (no GradScaler path)
        report_to="none",
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )
    
    # Setup callbacks
    callbacks = []
    
    # Note: Best model saving is handled by TrainingArguments with load_best_model_at_end=True
    # The Trainer will automatically save checkpoints and load the best one at the end
    if len(eval_examples) > 0:
        callbacks.append(EvalVotingCallback(eval_examples, tokenizer, num_samples=3))
        print("✅ Evaluation callback added")
        print("✅ Best model will be saved automatically via TrainingArguments")
    else:
        print("⚠️  No eval examples - best model checkpoint disabled")
    
    # Pre-training evaluation
    if len(eval_examples) > 0:
        print("\n" + "=" * 60)
        print("PRE-TRAINING EVALUATION")
        print("=" * 60)
        acc_before, f1_before, _, _ = evaluate_with_voting(model, tokenizer, eval_examples[:12], num_samples=1)
        print(f"Before training: accuracy={acc_before:.4f}, macro_f1={f1_before:.4f}")
    
    # Validate training data before creating trainer
    print("\n" + "=" * 60)
    print("VALIDATING TRAINING DATA")
    print("=" * 60)
    
    # Check for invalid token IDs in the dataset
    vocab_size = len(tokenizer) if hasattr(tokenizer, "__len__") else tokenizer.vocab_size
    max_token_id = max(max(ex["input_ids"]) for ex in train_examples[:10]) if train_examples else 0
    print(f"Vocabulary size: {vocab_size}")
    print(f"Max token ID in sample: {max_token_id}")
    
    if max_token_id >= vocab_size:
        print(f"⚠️  WARNING: Found token IDs >= vocab_size ({max_token_id} >= {vocab_size})")
        print("   This can cause CUDA assertion errors. Filtering invalid tokens...")
        # Filter out examples with invalid token IDs
        valid_examples = []
        for ex in train_examples:
            if all(0 <= tid < vocab_size for tid in ex["input_ids"]):
                valid_examples.append(ex)
        print(f"   Filtered {len(train_examples)} -> {len(valid_examples)} valid examples")
        train_examples = valid_examples
        train_dataset = Dataset.from_list(train_examples)
    
    # Create trainer with error handling
    try:
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=_collate_fn,
            callbacks=callbacks,
        )
    except RuntimeError as e:
        if "CUDA" in str(e) or "cuda" in str(e).lower():
            print(f"\n❌ CUDA Error during trainer initialization:")
            print(f"   {e}")
            print(f"\n💡 Troubleshooting steps:")
            print(f"   1. Set ENABLE_CUDA_DEBUG = True in config cell for better error reporting")
            print(f"   2. Restart kernel and try again")
            print(f"   3. Reduce batch size or max_length if memory is an issue")
            print(f"   4. Check for invalid token IDs in training data")
            raise
        else:
            raise
    
    # Train with error handling
    print("\n" + "=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    
    try:
        trainer.train()
    except RuntimeError as e:
        if "CUDA" in str(e) or "cuda" in str(e).lower() or "device-side assert" in str(e).lower():
            print(f"\n❌ CUDA Error during training:")
            print(f"   {e}")
            print(f"\n💡 Troubleshooting steps:")
            print(f"   1. Set ENABLE_CUDA_DEBUG = True in config cell and restart kernel")
            print(f"   2. Check training data for invalid token IDs or labels")
            print(f"   3. Reduce batch size: TRAIN_BATCH_SIZE = 1")
            print(f"   4. Reduce max_length: TRAIN_MAX_LENGTH = 512")
            print(f"   5. Try CPU training: Set device to CPU if GPU issues persist")
            
            # Try to save partial progress
            try:
                print(f"\n💾 Attempting to save checkpoint before exit...")
                trainer.save_model(TRAINING_OUTPUT_DIR + "_partial")
                print(f"   Saved to: {TRAINING_OUTPUT_DIR}_partial")
            except:
                pass
            
            raise
        else:
            raise
    
    # Post-training evaluation
    if len(eval_examples) > 0:
        print("\n" + "=" * 60)
        print("POST-TRAINING EVALUATION")
        print("=" * 60)
        acc_after, f1_after, preds, gold = evaluate_with_voting(model, tokenizer, eval_examples[:12], num_samples=1)
        print(f"After training: accuracy={acc_after:.4f}, macro_f1={f1_after:.4f}")
        print(f"Improvement: accuracy +{acc_after - acc_before:.4f}, F1 +{f1_after - f1_before:.4f}")
    
    # Save model (to Drive on Colab, local on Mac/Linux)
    print(f"\n{'=' * 60}")
    print("SAVING MODEL")
    print(f"{'=' * 60}")
    
    final_model_dir = Path(TRAINING_OUTPUT_DIR) / "final_model"
    best_model_dir = Path(TRAINING_OUTPUT_DIR) / "best_model"
    final_model_dir.mkdir(parents=True, exist_ok=True)
    best_model_dir.mkdir(parents=True, exist_ok=True)
    
    # Clear CUDA cache before saving to avoid errors
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        except RuntimeError:
            print("⚠️  CUDA error during cache clear - continuing anyway")
    
    # Save final model (after training completes)
    print(f"\n💾 Saving final model to: {final_model_dir}")
    final_saved = False
    try:
        # Move model to CPU before saving to avoid CUDA errors
        if hasattr(model, 'cpu'):
            model_cpu = model.cpu()
            original_model = trainer.model
            trainer.model = model_cpu
        
        trainer.save_model(str(final_model_dir), safe_serialization=True)
        tokenizer.save_pretrained(str(final_model_dir), safe_serialization=True)
        final_saved = True
        print(f"✅ Final model saved successfully!")
        
        # Restore original model
        if hasattr(model, 'cpu'):
            trainer.model = original_model
    except (RuntimeError, TypeError, Exception) as e:
        if "CUDA" in str(e) or "cuda" in str(e).lower() or "device-side assert" in str(e).lower():
            print(f"⚠️  CUDA error during final model save: {e}")
            print("   Attempting to save with model on CPU...")
            try:
                if hasattr(model, 'cpu'):
                    model_cpu = model.cpu()
                    trainer.model = model_cpu
                trainer.save_model(str(final_model_dir), safe_serialization=True)
                tokenizer.save_pretrained(str(final_model_dir), safe_serialization=True)
                final_saved = True
                print(f"✅ Final model saved successfully (CPU mode)!")
            except Exception as e2:
                print(f"❌ Failed to save final model: {e2}")
        else:
            print(f"⚠️  Error saving final model: {e}")
            try:
                trainer.save_model(str(final_model_dir))
                tokenizer.save_pretrained(str(final_model_dir))
                final_saved = True
                print(f"✅ Final model saved (fallback method)")
            except Exception as e2:
                print(f"❌ Failed to save final model: {e2}")
    
    # Save best model (from checkpoint)
    print(f"\n💾 Saving best model to: {best_model_dir}")
    best_saved = False
    best_checkpoint = None
    
    try:
        # Find best checkpoint
        if checkpoint_dir.exists():
            checkpoint_files = list(checkpoint_dir.glob("checkpoint-*"))
            if checkpoint_files:
                checkpoint_dirs = [d for d in checkpoint_files if d.is_dir()]
                if checkpoint_dirs:
                    best_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.name.split("-")[1]) if len(x.name.split("-")) > 1 and x.name.split("-")[1].isdigit() else 0)
                    print(f"   Found checkpoint: {best_checkpoint.name}")
        
        if best_checkpoint and best_checkpoint.exists():
            import shutil
            adapter_files = list(best_checkpoint.glob("adapter*"))
            if adapter_files:
                for file in best_checkpoint.glob("*"):
                    if file.is_file():
                        shutil.copy2(file, best_model_dir / file.name)
                print(f"✅ Best model (PEFT adapter) saved!")
                best_saved = True
            else:
                try:
                    trainer.save_model(str(best_model_dir), safe_serialization=True)
                    tokenizer.save_pretrained(str(best_model_dir), safe_serialization=True)
                    best_saved = True
                    print(f"✅ Best model saved!")
                except Exception as e:
                    print(f"⚠️  Error saving best model: {e}")
        else:
            print("⚠️  No checkpoint found - using final model as best model")
            if final_saved:
                import shutil
                for file in final_model_dir.glob("*"):
                    if file.is_file():
                        shutil.copy2(file, best_model_dir / file.name)
                best_saved = True
                print(f"✅ Copied final model to best_model directory")
    except Exception as e:
        print(f"⚠️  Error saving best model: {e}")
    
    # If final model save failed, try to load from best checkpoint
    if not final_saved and best_saved:
        print(f"\n💡 Final model save failed, but best model is available.")
        print(f"   You can load the best model from: {best_model_dir}")
        if best_checkpoint:
            print(f"   Or continue training from checkpoint: {best_checkpoint}")
    
    # Summary
    print(f"\n{'=' * 60}")
    print("MODEL SAVE SUMMARY")
    print(f"{'=' * 60}")
    print(f"Final model: {'✅ Saved' if final_saved else '❌ Failed'} → {final_model_dir}")
    print(f"Best model:  {'✅ Saved' if best_saved else '❌ Failed'} → {best_model_dir}")
    print(f"Checkpoints: {checkpoint_dir}")
    
    if IN_COLAB:
        print(f"\n📁 All models saved to Google Drive - will persist after Colab session ends")
        print(f"💡 To load best model: model = AutoModelForCausalLM.from_pretrained('{best_model_dir}')")
        print(f"💡 To load final model: model = AutoModelForCausalLM.from_pretrained('{final_model_dir}')")
    
    # Update global model to use trained model for evaluation
    _granite_model = model
    _granite_tokenizer = tokenizer
    print("\n✅ Training complete! Model is ready for evaluation.")
    
else:
    print("⏭️  Training skipped (TRAIN_MODEL = False)")
    print("   Using base model for evaluation.")

TRAINING GRANITE MODEL WITH PEFT/LoRA
✅ CUDA cache cleared successfully
Cleared GPU/MPS/CPU caches.
Using model and tokenizer already loaded from checkpoint (skipping reload).
✅ Gradient checkpointing enabled

LOADING RATIONALE CSV
Source: /content/drive/MyDrive/granite_clarity/data/qevasion_rationale_dataset_20260204_163024.csv
Loaded 118 rationale rows with verdict_match=True

Building training examples...
✅ Built 118 training examples (vocab_size=49155)
✅ Built 118 training examples

Loading eval examples...
✅ Loaded 60 eval examples
✅ Evaluation callback added
✅ Best model will be saved automatically via TrainingArguments

PRE-TRAINING EVALUATION


Evaluating:   0%|          | 0/12 [00:00<?, ?example/s]

Before training: accuracy=0.3333, macro_f1=0.2762

VALIDATING TRAINING DATA
Vocabulary size: 49155
Max token ID in sample: 49153

STARTING TRAINING


Step,Training Loss
10,1.173513
20,1.226285
30,1.167058
40,1.198092
50,1.151111
60,1.232919



POST-TRAINING EVALUATION


Evaluating:   0%|          | 0/12 [00:00<?, ?example/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


After training: accuracy=0.2500, macro_f1=0.1333
Improvement: accuracy +-0.0833, F1 +-0.1429

SAVING MODEL
⚠️  CUDA error during cache clear - continuing anyway

💾 Saving final model to: /content/drive/MyDrive/granite_clarity/models/granite_clarity_finetuned/final_model
⚠️  CUDA error during final model save: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

   Attempting to save with model on CPU...
❌ Failed to save final model: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


💾 Saving best model to: /content/drive/MyDrive/granite_clarity/models/granite_clarity_finetuned/best_model
   Found checkpoint: checkpoint-60
✅

In [11]:
# Initialize self-consistency strategy
strategy = GraniteSelfConsistencyStrategy(
    samples=SAMPLES,
    temperature=TEMPERATURE
)

print("✅ Strategy initialized!")

✅ Strategy initialized!


## Helper Functions

In [12]:
def map_to_clarity_format(label: str) -> str:
    """Map internal labels to CLARITY submission format."""
    mapping = {
        "Direct Reply": "Direct Reply",
        "Direct Non-Reply": "Direct Non-Reply",
        "Indirect": "Indirect",
        "Clear Reply": "Direct Reply",
        "Clear Non-Reply": "Direct Non-Reply",
        "Ambivalent Reply": "Indirect",
        "Ambivalent": "Indirect",
    }
    return mapping.get(label, "Indirect")

## Evaluation

Evaluate the Granite model on the QEvasion test set. Run the cells below for metrics (accuracy, macro F1, confusion matrix, classification report).


### Evaluate on QEvasion Test Set

(1) Load balanced test data, (2) run predictions with voting, (3) **RESULTS** cell prints all metrics.

In [31]:
# Load test/eval data: EVAL_INPUT_CSV if set, else dataset/qevasion_test_308.csv if present, else HuggingFace
import os
print("=" * 60)
print("EVALUATION ON QEVASION TEST SET")
print("=" * 60)

eval_csv = EVAL_INPUT_CSV or (DEFAULT_EVAL_CSV if os.path.isfile(DEFAULT_EVAL_CSV) else None)
if eval_csv:
    print(f"Using saved CSV: {eval_csv}")
    examples, true_labels = load_eval_from_csv(eval_csv, samples_per_label=TEST_SAMPLES_PER_LABEL)
else:
    print("Using HuggingFace QEvasion test split")
    examples, true_labels = load_balanced_test_data(
        split="test",
        samples_per_label=TEST_SAMPLES_PER_LABEL
    )

# Limit samples if requested
if MAX_TEST_SAMPLES and len(examples) > MAX_TEST_SAMPLES:
    print(f"Limiting evaluation to {MAX_TEST_SAMPLES} samples")
    examples = examples[:MAX_TEST_SAMPLES]
    true_labels = true_labels[:MAX_TEST_SAMPLES]

# Sample 2 random examples per label and print prompts BEFORE interleaving
print("\n" + "=" * 60)
print("SAMPLE EXAMPLES WITH PROMPTS (2 per label)")
print("=" * 60)

from collections import defaultdict
import random

# Group examples by label for sampling
examples_by_label = defaultdict(list)
for ex, label in zip(examples, true_labels):
    examples_by_label[label].append(ex)

# Create temporary strategy just for prompt building
strategy_temp = GraniteClarityStrategy()

# Sample 2 random examples per label and show prompts
for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
    if label in examples_by_label and len(examples_by_label[label]) >= 2:
        sampled = random.sample(examples_by_label[label], 2)
        print(f"\n--- {label} Examples ---")
        for idx, example in enumerate(sampled, 1):
            question = example.get("question", "")
            answer = example.get("answer", "")
            prompt = strategy_temp.build_prompt(question, answer)
            print(f"\nExample {idx}:")
            print(f"Question: {question[:150]}...")
            print(f"Answer: {answer[:150]}...")
            print(f"\nPrompt:")
            print("-" * 60)
            print(prompt)
            print("-" * 60)

# Reorganize to alternate between labels (handle imbalance)
label_groups = defaultdict(list)
for ex, label in zip(examples, true_labels):
    label_groups[label].append((ex, label))

# Interleave samples from each label group
examples_interleaved = []
true_labels_interleaved = []
max_per_label = max(len(items) for items in label_groups.values())
for i in range(max_per_label):
    for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
        if i < len(label_groups[label]):
            ex, lbl = label_groups[label][i]
            examples_interleaved.append(ex)
            true_labels_interleaved.append(lbl)

examples = examples_interleaved
true_labels = true_labels_interleaved

print(f"Evaluating on {len(examples)} examples (interleaved by label)...")
print(f"Label distribution: {dict(Counter(true_labels))}")

EVALUATION ON QEVASION TEST SET
Loading test split from ailsntua/QEvasion...
Loaded 308 examples from test split
Label distribution before balancing: {'Indirect': 206, 'Direct Reply': 79, 'Direct Non-Reply': 23}
Label distribution after balancing: {'Indirect': 23, 'Direct Reply': 23, 'Direct Non-Reply': 23}
Total balanced samples: 69

SAMPLE EXAMPLES WITH PROMPTS (2 per label)

--- Direct Reply Examples ---

Example 1:
Question: Q. Mr. President, on the way to Europe, you gave a very interesting interview for the Times newspaper in which you basically said that you regret your...
Answer: I don't regret it at all. Removing Saddam Hussein made the world a safer place. And yes, I told the guy— the guy said, Now what could you do over? Fir...

Prompt:
------------------------------------------------------------
You are analyzing political interview answers for clarity classification.

Question: Q. Mr. President, on the way to Europe, you gave a very interesting interview for the Times news

In [32]:
# Run predictions with per-sample feedback
predictions = []
pred_labels = []
pred_labels_clarity = []
true_labels_clarity = []

print("\n" + "=" * 60)
print("RUNNING PREDICTIONS WITH PER-SAMPLE FEEDBACK")
print("=" * 60)

for idx, (example, true_label) in enumerate(zip(examples, true_labels), start=1):
    # Get prediction
    question = example.get("question", "")
    answer = example.get("answer", "")
    
    if not question or not answer:
        pred = {"label": "Indirect", "reasoning": ["Missing question or answer"], "votes": {"Indirect": SAMPLES}, "all_predictions": ["Indirect"] * SAMPLES}
    else:
        pred = strategy.predict_single_with_voting(question, answer)
    
    predictions.append(pred)
    
    # Extract and map labels
    pred_label = pred["label"]
    pred_labels.append(pred_label)
    pred_label_clarity = map_to_clarity_format(pred_label)
    pred_labels_clarity.append(pred_label_clarity)
    true_label_clarity = map_to_clarity_format(true_label)
    true_labels_clarity.append(true_label_clarity)
    
    # Check if correct
    is_correct = pred_label_clarity == true_label_clarity
    status = "✅ CORRECT" if is_correct else "❌ WRONG"
    
    # Display result
    print(f"\n[{idx+1}/{len(examples)}] {status}")
    print(f"  Question: {question[:110]}...")
    print(f"  True Label: {true_label_clarity}")
    print(f"  Predicted Label: {pred_label_clarity}")
    print(f"  Votes: {pred.get('votes', {})}")
    if not is_correct:
        print(f"  ⚠️  Expected '{true_label_clarity}' but got '{pred_label_clarity}'")

print(f"\n✅ Predictions completed for {len(predictions)} examples")

Attempting to cast a BatchEncoding to type None. This is not supported.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attemptin


RUNNING PREDICTIONS WITH PER-SAMPLE FEEDBACK
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different

Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not sup

Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking a

Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not sup

...

Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

[59/69] ❌ WRONG
  Question: Q. Have you discussed proliferation of weapons of mass destruction and missile delivery, and what are the resu...
  True Label: Direct Reply
  Predicte

Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not sup

Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

[64/69] ✅ CORRECT
  Question: Q. Mr. President, are you disappointed that the Israelis and the Palestinians haven't made more specific progr...
  True Label: Indirect
  Predicted Label: Indirect
  Votes: {'Indirect': 3}
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on 

Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.
Attempting to cast a BatchEncoding to type None. This is not supported.


Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

[69/69] ❌ WRONG
  Question: Q. ——there are many Democrats, as well as some medical experts, who say that the abstinence provision—spending...
  True Label: Direct Non-Reply
  Predicted Label: Indirect
  Votes: {'Indirect': 3}
  ⚠️  Expected 'Direct Non-Reply' but got 'Indirect'
Batch generation failed, falling back to sequential: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)
Error in predict_single: Expected all tensors to be on the same device, but got index is on cpu, d

In [15]:
# Compute metrics
accuracy = accuracy_score(true_labels_clarity, pred_labels_clarity)
macro_f1 = f1_score(true_labels_clarity, pred_labels_clarity, average="macro")
per_class_f1 = f1_score(true_labels_clarity, pred_labels_clarity, average=None, labels=["Direct Reply", "Direct Non-Reply", "Indirect"])

# Per-label accuracy
from collections import Counter
label_accuracies = {}
for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
    label_mask = [tl == label for tl in true_labels_clarity]
    if any(label_mask):
        label_true = [tl for tl, m in zip(true_labels_clarity, label_mask) if m]
        label_pred = [pl for pl, m in zip(pred_labels_clarity, label_mask) if m]
        label_accuracies[label] = accuracy_score(label_true, label_pred)

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"\nPer-class F1:")
print(f"  Direct Reply: {per_class_f1[0]:.4f}")
print(f"  Direct Non-Reply: {per_class_f1[1]:.4f}")
print(f"  Indirect: {per_class_f1[2]:.4f}")
print(f"\nPer-label Accuracy:")
for label, acc in label_accuracies.items():
    print(f"  {label}: {acc:.4f}")


RESULTS
Overall Accuracy: 0.3333
Macro F1: 0.1667

Per-class F1:
  Direct Reply: 0.0000
  Direct Non-Reply: 0.0000
  Indirect: 0.5000

Per-label Accuracy:
  Direct Reply: 0.0000
  Direct Non-Reply: 0.0000
  Indirect: 1.0000


In [16]:
# Confusion Matrix
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(true_labels_clarity, pred_labels_clarity, labels=["Direct Reply", "Direct Non-Reply", "Indirect"])

print("Labels: [Direct Reply, Direct Non-Reply, Indirect]")
print(cm)

CONFUSION MATRIX
Labels: [Direct Reply, Direct Non-Reply, Indirect]
[[ 0  0 23]
 [ 0  0 23]
 [ 0  0 23]]


In [17]:
# Classification Report
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(true_labels_clarity, pred_labels_clarity, labels=["Direct Reply", "Direct Non-Reply", "Indirect"]))

CLASSIFICATION REPORT
                  precision    recall  f1-score   support

    Direct Reply       0.00      0.00      0.00        23
Direct Non-Reply       0.00      0.00      0.00        23
        Indirect       0.33      1.00      0.50        23

        accuracy                           0.33        69
       macro avg       0.11      0.33      0.17        69
    weighted avg       0.11      0.33      0.17        69



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:
# Label Distribution
print("LABEL DISTRIBUTION")
print("=" * 60)
print("True labels:", dict(Counter(true_labels_clarity)))
print("Predicted labels:", dict(Counter(pred_labels_clarity)))

LABEL DISTRIBUTION
True labels: {'Direct Reply': 23, 'Direct Non-Reply': 23, 'Indirect': 23}
Predicted labels: {'Indirect': 69}


In [19]:
# Detailed Sample Predictions with Correctness
print("\n" + "=" * 60)
print("DETAILED SAMPLE PREDICTIONS")
print("=" * 60)

# Show examples from each label category
label_indices = {"Direct Reply": [], "Direct Non-Reply": [], "Indirect": []}
for i, label in enumerate(true_labels_clarity):
    if len(label_indices[label]) < 2:  # Show 2 examples per label
        label_indices[label].append(i)

for label_type, indices in label_indices.items():
    print(f"\n--- {label_type} Examples ---")
    for i in indices:
        is_correct = pred_labels_clarity[i] == true_labels_clarity[i]
        status = "✅ CORRECT" if is_correct else "❌ WRONG"
        print(f"\nExample {i+1} [{status}]:")
        print(f"  Question: {examples[i]['question'][:120]}...")
        print(f"  Answer: {examples[i]['answer'][:120]}...")
        print(f"  True Label: {true_labels_clarity[i]}")
        print(f"  Predicted Label: {pred_labels_clarity[i]}")
        print(f"  Votes: {predictions[i].get('votes', {})}")
        print(f"  All Predictions: {predictions[i].get('all_predictions', [])}")
        if predictions[i].get('reasoning'):
            print(f"  Reasoning (first sample): {predictions[i]['reasoning'][0][:250]}...")


DETAILED SAMPLE PREDICTIONS

--- Direct Reply Examples ---

Example 1 [❌ WRONG]:
  Question: Q. Was this coordinated with you? And are you willing to speak to President Asad to end the crisis in Lebanon?...
  Answer: No, it wasn't coordinated with me, and my patience ran out on President Asad a long time ago. And the reason why is, is ...
  True Label: Direct Reply
  Predicted Label: Indirect
  Votes: {'Indirect': 3}
  All Predictions: ['Indirect', 'Indirect', 'Indirect']
  Reasoning (first sample): Prediction failed...

Example 4 [❌ WRONG]:
  Question: Q. You said yesterday in your statement that the North Korean nuclear test was unacceptable. Your chief negotiator for t...
  Answer: That's a fair question. First of all, I am making it clear our policy hasn't changed. It's important for the folks to un...
  True Label: Direct Reply
  Predicted Label: Indirect
  Votes: {'Indirect': 3}
  All Predictions: ['Indirect', 'Indirect', 'Indirect']
  Reasoning (first sample): Prediction failed

## Generate CLARITY Submission

In [23]:
# Load CLARITY evaluation data
print("GENERATING CLARITY SUBMISSION")
print("=" * 60)

examples_eval, indices = load_clarity_eval_data(EVAL_FILE)

print(f"Generating predictions for {len(examples_eval)} examples...")

GENERATING CLARITY SUBMISSION
Loading CLARITY evaluation dataset from /Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv...


FileNotFoundError: Evaluation file not found: /Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv

In [ ]:
# Run predictions on evaluation set
predictions_eval = strategy.predict_batch(examples_eval)

# Extract labels and map to CLARITY format
pred_labels_eval = [map_to_clarity_format(pred["label"]) for pred in predictions_eval]

# Validate labels
valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
for i, label in enumerate(pred_labels_eval):
    if label not in valid_labels:
        print(f"Warning: Invalid label '{label}' at index {i}, defaulting to 'Indirect'")
        pred_labels_eval[i] = "Indirect"

print(f"✅ Predictions completed for evaluation set")

In [ ]:
# Create submission tuple
submission = (indices, pred_labels_eval)

# Save pickle
print(f"Saving submission to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(submission, f)

print(f"✅ Submission saved: {OUTPUT_FILE}")
print(f"   Samples: {len(indices)}")
print(f"   Predictions: {len(pred_labels_eval)}")
print(f"   Label distribution: {dict(Counter(pred_labels_eval))}")

In [ ]:
# Validate submission
print(f"Validating submission file: {OUTPUT_FILE}")

with open(OUTPUT_FILE, 'rb') as f:
    indices_check, predictions_check = pickle.load(f)

# Basic validation
assert len(indices_check) == len(predictions_check), "Indices and predictions length mismatch"
assert all(isinstance(idx, int) for idx in indices_check), "All indices must be integers"
assert all(pred in ["Direct Reply", "Direct Non-Reply", "Indirect"] for pred in predictions_check), "Invalid prediction labels"

print("✅ Submission file validation passed!")
print(f"   - {len(indices_check)} predictions")
print(f"   - Indices range: {min(indices_check)} to {max(indices_check)}")
print(f"   - Unique predictions: {set(predictions_check)}")

## Summary

Evaluation complete! The submission file is ready for CLARITY-SemEval-2026.